In [1]:
# rag_pipeline.py
from sentence_transformers import CrossEncoder


In [2]:
reranker = CrossEncoder("BAAI/bge-reranker-base")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
def rerank(question, docs, top_k=3):
    # 1. 질문이랑 각 문서를 '쌍'으로 묶기
    pairs = [(question,doc) for doc in docs]                          # [(질문, 문서1), (질문, 문서2), ...]
    # 2. 각 쌍의 관련도 점수 매기기
    scores = reranker.predict(pairs)     # 점수 배열
    # 3. 점수 높은 순으로 문서 정렬 → 상위 top_k개 반환
    scored = list(zip(scores, docs))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:top_k]]




In [4]:
from rag_app import build_vectorstore, TEXTS

vectorstore = build_vectorstore(TEXTS)

def retrieve_and_rerank(question, first_k=20, top_k=5):
    # 1. 검색: 후보를 넉넉히 (first_k개)
    candidates = vectorstore.similarity_search(question, k=first_k)   # Document 리스트
    # 2. 재정렬: candidates에서 상위 top_k로 좁히기
    texts = [d.page_content for d in candidates]
    reranked = rerank(question, texts, top_k)
    return reranked



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
vectorstore.delete_collection()      # 옛 컬렉션 삭제
# 또는 persist 폴더 통째로: import shutil; shutil.rmtree("경로", ignore_errors=True)
vectorstore = build_vectorstore(TEXTS)   # 깨끗하게 재빌드
print(vectorstore._collection.count())   # 이제 26 나와야 정상

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

26


In [6]:
from rag_app import TEXTS
import rag_app, inspect

print("1) len(TEXTS):", len(TEXTS))
print("2) TEXTS 샘플:", TEXTS[:2])
print("3) collection count:", vectorstore._collection.count())
print("4) build_vectorstore:\n", inspect.getsource(rag_app.build_vectorstore))

1) len(TEXTS): 26
2) TEXTS 샘플: ['RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.', '환각(hallucination)은 모델이 학습하지 않았거나 근거 없는 내용을 사실인 것처럼 그럴듯하게 지어내는 현상이다.']
3) collection count: 26
4) build_vectorstore:
 def build_vectorstore(texts):
    documents = [Document(page_content=t) for t in texts]

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(documents)

    # 한국어 문서라 multilingual 임베딩 (English 전용 X)
    embeddings = HuggingFaceEmbeddings(
        model_name="paraphrase-multilingual-MiniLM-L12-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,          # 파라미터명은 embedding (단수)
        collection_name="rag_app",
    )
    return vectorstore



In [7]:
q = "RAG는 환각을 어떻게 줄이는가?"

# 재정렬 전 (그냥 검색 순서)
before = [d.page_content for d in vectorstore.similarity_search(q, k=5)]
# 재정렬 후
after = retrieve_and_rerank(q, first_k=20, top_k=3)

print("=== 검색 순서 (before) ===")
for i, t in enumerate(before, 1):
    print(f"{i}. {t}")
print("\n=== 재정렬 후 (after) ===")
for i, t in enumerate(after, 1):
    print(f"{i}. {t}")

=== 검색 순서 (before) ===
1. RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.
2. 청크를 나눌 때 일부를 겹치게(overlap) 하면 경계에서 잘린 문맥 손실을 줄일 수 있다.
3. LoRA는 원래 가중치는 얼리고 작은 저랭크 행렬만 학습해 파인튜닝 비용을 크게 줄인다.
4. 파인튜닝은 지식 주입보다 말투·형식·도메인 적응에 강하고, 최신 사실 반영에는 RAG가 낫다.
5. RAG는 모델 가중치를 바꾸지 않고 외부 지식을 주입하므로 지식 갱신이 잦은 도메인에 적합하다.

=== 재정렬 후 (after) ===
1. RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.
2. 파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 바꾸는 것이다.
3. 환각(hallucination)은 모델이 학습하지 않았거나 근거 없는 내용을 사실인 것처럼 그럴듯하게 지어내는 현상이다.


In [8]:
q = "RAG는 환각을 어떻게 줄이는가?"
cands = [d.page_content for d in vectorstore.similarity_search(q, k=20)]
scores = reranker.predict([(q, d) for d in cands])
for s, d in sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)[:8]:
    print(f"{s:8.3f}  {d[:45]}")

   0.006  RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답
   0.001  파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 바꾸는 것
   0.001  환각(hallucination)은 모델이 학습하지 않았거나 근거 없는 내용을 사실
   0.001  청크를 나눌 때 일부를 겹치게(overlap) 하면 경계에서 잘린 문맥 손실을 줄
   0.001  cross-encoder는 질문과 문서를 함께 입력해 상호작용을 보고 관련도를 매
   0.000  bi-encoder는 질문과 문서를 각각 따로 임베딩해 유사도를 재므로 빠르지만 
   0.000  청크 크기가 너무 작으면 문장이 토막나 검색 품질이 떨어지고, 너무 크면 잡내용이
   0.000  RAG에서 검색 품질이 나쁘면 엉뚱한 문서가 프롬프트에 들어가 오히려 환각이 늘 


In [9]:
test_qs = [
    "적은 메모리로 큰 모델을 학습시키려면?",          # 정답 QLoRA("4비트+VRAM") / 질문은 '메모리'·'학습' → 단어 어긋남
    "긴 문서를 어떻게 잘라야 검색이 잘 되나?",          # 정답 청킹/overlap / '자르다'vs'청크' 단어 어긋남
    "답변이 근거 있는지 어떻게 확인하나?",              # 정답 faithfulness / '근거 확인'vs'faithfulness' 어긋남
    "두 문장이 의미가 비슷한지 어떻게 판단하나?",        # 정답 임베딩/코사인 / 단어 다름
    "빠른 검색이랑 정확한 검색을 둘 다 잡으려면?",        # 정답 2단계 검색 / 단어 어긋남
]

for q in test_qs:
    before = [d.page_content for d in vectorstore.similarity_search(q, k=5)]
    after  = retrieve_and_rerank(q, first_k=20, top_k=3)
    moved = before[0] != after[0]          # 1위가 바뀌었나
    print(f"\n{'★ 순위변동' if moved else '  변동없음'}  | Q: {q}")
    print(f"  before 1위: {before[0][:40]}")
    print(f"  after  1위: {after[0][:40]}")


★ 순위변동  | Q: 적은 메모리로 큰 모델을 학습시키려면?
  before 1위: 파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 
  after  1위: 파인튜닝은 지식 주입보다 말투·형식·도메인 적응에 강하고, 최신 사실 반

  변동없음  | Q: 긴 문서를 어떻게 잘라야 검색이 잘 되나?
  before 1위: 청크를 나눌 때 일부를 겹치게(overlap) 하면 경계에서 잘린 문맥 
  after  1위: 청크를 나눌 때 일부를 겹치게(overlap) 하면 경계에서 잘린 문맥 

★ 순위변동  | Q: 답변이 근거 있는지 어떻게 확인하나?
  before 1위: 프롬프트에 '컨텍스트에 없으면 없다고 답하라'는 지시를 넣으면 근거 없는
  after  1위: RAGAS의 faithfulness는 생성된 답이 검색된 컨텍스트에 근거

★ 순위변동  | Q: 두 문장이 의미가 비슷한지 어떻게 판단하나?
  before 1위: 의미가 비슷한 문장은 임베딩 공간에서 가까이, 무관한 문장은 멀리 위치한
  after  1위: temperature가 높을수록 생성이 다양해지고 낮을수록 결정적(det

★ 순위변동  | Q: 빠른 검색이랑 정확한 검색을 둘 다 잡으려면?
  before 1위: RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 
  after  1위: 2단계 검색은 bi-encoder로 후보를 넓게 뽑고 cross-enco


In [10]:
q = "답변이 근거 있는지 어떻게 확인하나?"
before = [d.page_content for d in vectorstore.similarity_search(q, k=5)]
after  = retrieve_and_rerank(q, first_k=20, top_k=3)
print("=== before (bi-encoder) ===")
for i,t in enumerate(before,1): print(f"{i}. {t}")
print("\n=== after (re-ranked) ===")
for i,t in enumerate(after,1):  print(f"{i}. {t}")

=== before (bi-encoder) ===
1. 프롬프트에 '컨텍스트에 없으면 없다고 답하라'는 지시를 넣으면 근거 없는 답을 줄일 수 있다.
2. 청크를 나눌 때 일부를 겹치게(overlap) 하면 경계에서 잘린 문맥 손실을 줄일 수 있다.
3. 의미가 비슷한 문장은 임베딩 공간에서 가까이, 무관한 문장은 멀리 위치한다.
4. RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.
5. 파인튜닝은 지식 주입보다 말투·형식·도메인 적응에 강하고, 최신 사실 반영에는 RAG가 낫다.

=== after (re-ranked) ===
1. RAGAS의 faithfulness는 생성된 답이 검색된 컨텍스트에 근거하는지를 측정한다.
2. RAG에서 검색 품질이 나쁘면 엉뚱한 문서가 프롬프트에 들어가 오히려 환각이 늘 수 있다.
3. 환각(hallucination)은 모델이 학습하지 않았거나 근거 없는 내용을 사실인 것처럼 그럴듯하게 지어내는 현상이다.


In [11]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model="openai/gpt-oss-120b")   # 생성용 (또는 gpt-oss-120b)
prompt = ChatPromptTemplate.from_template(
    """아래 컨텍스트를 근거로만 질문에 답하라. 없으면 없다고 답하라. 한국어로 답하라.

컨텍스트:
{context}

질문: {question}

답변:"""
)

def generate_answer(question, reranked_docs):
    context = "\n\n".join(reranked_docs)   # 재정렬된 문서들을 하나의 문자열로
    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    return response.content

In [12]:
def rag_pipeline(question):
    reranked = retrieve_and_rerank(question, first_k=20, top_k=3)   # 검색+재정렬
    answer = generate_answer(question, reranked)                    # 생성
    return answer, reranked   # 답 + 근거(평가에 쓸 거)

In [13]:
q = "RAG는 환각을 어떻게 줄이는가?"
answer, docs = rag_pipeline(q)
print("답:", answer)
print("근거:", docs)

답: RAG는 질문에 대한 답변을 생성할 때, 먼저 관련 문서를 검색해 그 내용을 프롬프트에 포함시킵니다. 이렇게 실제 근거가 되는 문서를 함께 제공함으로써 모델이 근거 없이 추론하거나 사실이 아닌 내용을 만들어내는 환각을 억제하고, 최신 정보를 답변에 반영할 수 있게 합니다.
근거: ['RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.', '파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 바꾸는 것이다.', '환각(hallucination)은 모델이 학습하지 않았거나 근거 없는 내용을 사실인 것처럼 그럴듯하게 지어내는 현상이다.']


In [15]:
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# judge (gpt-oss-120b로!)
judge_llm = LangchainLLMWrapper(ChatGroq(model="openai/gpt-oss-120b"))
judge_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
)



C:\Users\조영석\AppData\Local\Temp\ipykernel_8968\1472685704.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
C:\Users\조영석\AppData\Local\Temp\ipykernel_8968\1472685704.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatGroq(model="openai/gpt-oss-120b"))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\조영석\AppData\Local\Temp\ipykernel_8968\1472685704.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(


In [17]:
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import faithfulness   # TODO: context_precision 계열 추가
# judge_llm, judge_emb 는 위 셀에서 이미 정의됨 (재사용)
from ragas.metrics import faithfulness, LLMContextPrecisionWithoutReference

metrics = [faithfulness, LLMContextPrecisionWithoutReference()]

questions = [
    "답변이 근거 있는지 어떻게 확인하나?",
    "빠른 검색이랑 정확한 검색을 둘 다 잡으려면?",
    "긴 문서를 어떻게 잘라야 검색이 잘 되나?",
    "15년은 몇 일인가?",
]

def rag_no_rerank(question, k=3):
    docs = [d.page_content for d in vectorstore.similarity_search(question, k=k)]
    answer = generate_answer(question, docs)
    return answer, docs

def build_dataset(pipeline_fn):
    samples = []
    for q in questions:
        answer, docs = pipeline_fn(q)
        samples.append(SingleTurnSample(
            user_input=q, retrieved_contexts=docs, response=answer,
        ))
    return EvaluationDataset(samples=samples)

res_off = evaluate(dataset=build_dataset(rag_no_rerank), metrics=metrics, llm=judge_llm, embeddings=judge_emb)
res_on  = evaluate(dataset=build_dataset(rag_pipeline),  metrics=metrics, llm=judge_llm, embeddings=judge_emb)
print("재정렬 OFF:", res_off)
print("재정렬 ON :", res_on)

C:\Users\조영석\AppData\Local\Temp\ipykernel_8968\4252962996.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness   # TODO: context_precision 계열 추가
C:\Users\조영석\AppData\Local\Temp\ipykernel_8968\4252962996.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, LLMContextPrecisionWithoutReference
C:\Users\조영석\AppData\Local\Temp\ipykernel_8968\4252962996.py:4: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import 

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Exception raised in Job[3]: TimeoutError()


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Exception raised in Job[1]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[3]: TimeoutError()


재정렬 OFF: {'faithfulness': 0.3242, 'llm_context_precision_without_reference': 0.6667}
재정렬 ON : {'faithfulness': 0.5222, 'llm_context_precision_without_reference': nan}
